# nb18 — Distillation α-sweep (O1 → F17 upgrade test)

### Why this notebook exists

[findings.md](../docs/findings.md) currently lists **O1** as a single-point observation:
at α_distill = 0.7 the distilled HDC-RWKV student was *worse* than its NLL-only baseline.
We deliberately softened it because we only tested one α. The honest scrutiny note
states: "Cannot claim generality without sweeping α_distill."

This notebook closes that gap with a controlled 3-point sweep.

### Design

Single self-contained Kaggle run. One teacher trained inline, four students trained
in sequence with identical everything **except** the distillation coefficient:

| Run | α_nll | α_distill | Purpose |
|---|---|---|---|
| baseline | 1.0 | 0.0 | NLL-only reference (the floor to beat) |
| α=0.3 | 0.7 | 0.3 | Light distillation |
| α=0.5 | 0.5 | 0.5 | Balanced |
| α=0.7 | 0.3 | 0.7 | Heavy distillation (matches O1 config) |

All four students share: same seed, same teacher checkpoint, same architecture
(V=256, d=384, L=2), same 12k steps, same LR, same batch, same eval schedule.

### Decision rules

| Pattern | Verdict |
|---|---|
| All 3 distilled ≥ baseline BPC | **F17 strong negative**: cross-arch distill regresses HDC-RWKV across α. Upgrades O1 to a finding. |
| Monotonic increase in BPC with α | **F17 positive characterization**: regression scales with distill weight. |
| One α beats baseline, others worse | **F17 mixed**: optimal α exists but is fragile. Useful framing. |
| All 3 < baseline BPC | **O1 retraction**: distillation does help, our α=0.7 point was unlucky. |

### Runtime estimate

Teacher: ~6 min. Each student: ~10 min. Total: ~50 min on T4.

## Cell 1 — Setup

In [ ]:
import os, sys, subprocess
from pathlib import Path

os.chdir('/kaggle/working')
REPO_URL = 'https://github.com/elixpo/wozformer.git'
subprocess.run(['rm', '-rf', '/kaggle/working/wozformer'], check=True)
subprocess.run(['git', 'clone', REPO_URL, '/kaggle/working/wozformer'], check=True)
WOZFORMER_PATH = Path('/kaggle/working/wozformer')
os.chdir(WOZFORMER_PATH)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.'], check=True)

for m in list(sys.modules):
    if m.startswith('wozformer'):
        del sys.modules[m]

import wozformer as wz
import torch
import math
import copy
import matplotlib.pyplot as plt

device = wz.utils.get_device()
print(f'wozformer {wz.__version__}, device: {device}')

## Cell 2 — Hyperparameters

Student arch matches the O1 setup (V=256, d=384, L=2). Teacher matches the
nb13 small teacher (V=256, d=192, L=4, dropout=0.2 to prevent overfitting).

In [ ]:
VOCAB_SIZE = 256

# Teacher
T_D_MODEL  = 192
T_N_LAYERS = 4
T_N_HEADS  = 3
T_DROPOUT  = 0.2
T_STEPS    = 6000

# Student (HDC-RWKV)
S_D        = 384
S_LAYERS   = 2
S_STEPS    = 12000

# Shared
BATCH_SIZE = 32
BLOCK_SIZE = 64
LR         = 3e-3
EVAL_EVERY = 500
SEED       = 1337
DIST_TEMP  = 4.0

# Sweep points
SWEEP_ALPHAS = [0.0, 0.3, 0.5, 0.7]   # 0.0 = NLL-only baseline

RUN_DIR = Path('/kaggle/working/runs')
RUN_DIR.mkdir(parents=True, exist_ok=True)
TEACHER_PT = RUN_DIR / 'teacher_v256_d192_L4.pt'
BPE_JSON   = RUN_DIR / 'bpe_256.json'

## Cell 3 — Corpus + shared BPE tokenizer

Tokenizer is trained once and used by both teacher and all students. Critical for
the sweep to be valid — different tokenizers across runs would invalidate BPC comparison.

In [ ]:
wz.utils.set_seed(SEED)

text = wz.data.load_corpus(WOZFORMER_PATH / 'data' / 'tinyshakespeare.txt')
tok = wz.tokenizer.BPETokenizer.train(text, vocab_size=VOCAB_SIZE)
tok.save(BPE_JSON)

ids = torch.tensor(tok.encode(text), dtype=torch.long)
train_data, val_data = wz.data.split_train_val(ids)
print(f'tokens: train {len(train_data):,} / val {len(val_data):,}')

# Used in BPC conversion for all runs
sample_for_cpt = val_data[:5000].tolist()
avg_cpt = wz.metrics.avg_chars_per_token(tok, sample_for_cpt)
print(f'avg chars/token (shared for all BPC): {avg_cpt:.3f}')

## Cell 4 — Train the teacher (~6 min on T4)

Small dense transformer. Dropout 0.2 is essential — without it this overfits hard
on the ~430k-token tiny shakespeare corpus.

In [ ]:
wz.utils.set_seed(SEED)

t_cfg = wz.config.TransformerConfig(
    vocab_size=VOCAB_SIZE,
    d_model=T_D_MODEL,
    num_heads=T_N_HEADS,
    n_layers=T_N_LAYERS,
    mlp_mult=4,
    dropout=T_DROPOUT,
)
teacher = wz.models.TinyTransformer(t_cfg, block_size=BLOCK_SIZE).to(device)
print(f'teacher params: {wz.utils.count_params(teacher):,}')

t_train_cfg = wz.config.TrainConfig(
    batch_size=BATCH_SIZE, block_size=BLOCK_SIZE,
    lr=LR, n_steps=T_STEPS, eval_every=EVAL_EVERY,
    seed=SEED, weight_decay=0.0,
)
t_history, t_best = wz.trainer.train(
    teacher, train_data, val_data, t_train_cfg,
    device=device, eval_hard=False,
)
teacher_bpc = wz.metrics.bits_per_char(t_best['val'], avg_cpt)
print(f'\nteacher best val: {t_best["val"]:.4f} nats/token  (BPC {teacher_bpc:.4f})  at step {t_best["step"]}')

torch.save({
    'config': t_cfg.__dict__,
    'model_state': teacher.state_dict(),
    'best_val': t_best['val'],
    'best_bpc': teacher_bpc,
}, TEACHER_PT)
print(f'saved teacher → {TEACHER_PT}')

teacher.eval()
for p in teacher.parameters():
    p.requires_grad_(False)

## Cell 5 — The sweep: 4 students, same seed, same teacher

`α_distill = 0.0` is the NLL-only baseline; the rest use the teacher's soft targets.
The student is reinitialised fresh from the same seed each run so the only varying
factor is the loss coefficient.

In [ ]:
def run_student(alpha_distill: float, run_label: str):
    """Train one student. Returns (best_hard_val, bpc, history)."""
    wz.utils.set_seed(SEED)
    s_cfg = wz.config.HDCRWKVConfig(
        vocab_size=VOCAB_SIZE, d=S_D, n_layers=S_LAYERS, block_size=BLOCK_SIZE,
    )
    student = wz.models.HDCRWKV(s_cfg).to(device)

    s_train_cfg = wz.config.TrainConfig(
        batch_size=BATCH_SIZE, block_size=BLOCK_SIZE,
        lr=LR, n_steps=S_STEPS, eval_every=EVAL_EVERY,
        seed=SEED, weight_decay=0.0,
    )

    if alpha_distill == 0.0:
        # NLL-only baseline — no teacher involvement
        history, best = wz.trainer.train(
            student, train_data, val_data, s_train_cfg,
            device=device, eval_hard=True,
        )
    else:
        d_cfg = wz.config.DistillationConfig(
            temperature=DIST_TEMP,
            alpha_nll=1.0 - alpha_distill,
            alpha_distill=alpha_distill,
        )
        history, best = wz.distillation.train_with_distillation(
            student, teacher, train_data, val_data,
            s_train_cfg, d_cfg, device=device, eval_hard=True,
        )

    bpc = wz.metrics.bits_per_char(best['val'], avg_cpt)
    print(f'\n[{run_label}] best HARD val {best["val"]:.4f}  BPC {bpc:.4f}  step {best["step"]}')

    ckpt = RUN_DIR / f'student_alpha{alpha_distill:.1f}.pt'
    torch.save({
        'config': s_cfg.__dict__,
        'alpha_distill': alpha_distill,
        'model_state': student.state_dict(),
        'best_hard_val': best['val'],
        'bpc': bpc,
        'history': history,
        'best_step': best['step'],
    }, ckpt)
    return best['val'], bpc, history, student


results = {}
for alpha in SWEEP_ALPHAS:
    label = 'baseline (NLL-only)' if alpha == 0.0 else f'α_distill={alpha}'
    print(f'\n{"="*60}\n>>> {label}\n{"="*60}')
    val, bpc, hist, st = run_student(alpha, label)
    results[alpha] = {'val': val, 'bpc': bpc, 'history': hist, 'student': st}

print('\n\n=========== SWEEP SUMMARY ===========')
print(f'teacher BPC: {teacher_bpc:.4f}')
for alpha in SWEEP_ALPHAS:
    tag = 'baseline' if alpha == 0.0 else f'α={alpha}'
    r = results[alpha]
    print(f'  {tag:>12}: val {r["val"]:.4f}  BPC {r["bpc"]:.4f}')

## Cell 6 — Plot the sweep + verdict

In [ ]:
baseline_bpc = results[0.0]['bpc']
distilled_bpcs = {a: results[a]['bpc'] for a in SWEEP_ALPHAS if a > 0.0}

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

# Left: BPC vs alpha bar chart
xs = [str(a) if a > 0 else 'baseline\n(NLL-only)' for a in SWEEP_ALPHAS]
ys = [results[a]['bpc'] for a in SWEEP_ALPHAS]
colors = ['gray'] + ['steelblue']*3
bars = axes[0].bar(xs, ys, color=colors)
axes[0].axhline(baseline_bpc, color='gray', linestyle='--', alpha=0.6, label=f'baseline ({baseline_bpc:.3f})')
axes[0].axhline(teacher_bpc, color='green', linestyle='--', alpha=0.6, label=f'teacher ({teacher_bpc:.3f})')
axes[0].set_xlabel('α_distill'); axes[0].set_ylabel('BPC (lower = better)')
axes[0].set_title('Distillation α-sweep on HDC-RWKV student')
axes[0].legend(); axes[0].grid(alpha=0.3, axis='y')
for bar, val in zip(bars, ys):
    axes[0].text(bar.get_x() + bar.get_width()/2, val + 0.01, f'{val:.3f}',
                 ha='center', va='bottom', fontsize=9)

# Right: val curves overlaid
for alpha in SWEEP_ALPHAS:
    h = results[alpha]['history']
    steps = [hh[0] for hh in h]
    vhard = [hh[2] if len(hh) == 3 else hh[2] for hh in h]   # (step, vsoft, vhard) for distill; same for NLL-only
    label = 'baseline' if alpha == 0.0 else f'α={alpha}'
    axes[1].plot(steps, vhard, label=label, alpha=0.8)
axes[1].set_xlabel('step'); axes[1].set_ylabel('val(hard) nats/token')
axes[1].set_title('Training curves (hard val)')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()

# Verdict
deltas = {a: results[a]['bpc'] - baseline_bpc for a in distilled_bpcs}
all_worse  = all(d > 0 for d in deltas.values())
all_better = all(d < 0 for d in deltas.values())
monotonic_worse = (deltas[0.3] < deltas[0.5] < deltas[0.7])

print(f'\nbaseline (NLL-only) BPC: {baseline_bpc:.4f}')
for a, d in deltas.items():
    sign = '+' if d >= 0 else ''
    print(f'  α={a}: ΔBPC = {sign}{d:+.4f}  ({"worse" if d > 0 else "better"} than baseline)')

print('\n>>> VERDICT')
if all_worse and monotonic_worse:
    print('F17 STRONG NEGATIVE + monotonic: distillation regresses HDC-RWKV at every α,')
    print('and the regression scales with the distillation weight. Upgrades O1 to F17.')
elif all_worse:
    print('F17 STRONG NEGATIVE: distillation regresses HDC-RWKV at every α tested.')
    print('Upgrades O1 to F17 — the regression is not specific to α=0.7.')
elif all_better:
    print('O1 RETRACTION: distillation HELPS at every α tested. The original α=0.7')
    print('observation was a single unlucky data point.')
else:
    helpers = [a for a, d in deltas.items() if d < 0]
    hurters = [a for a, d in deltas.items() if d > 0]
    print(f'F17 MIXED: α∈{helpers} help, α∈{hurters} hurt.')
    print('Distillation works for HDC-RWKV but the α surface is non-trivial.')
    print('Useful framing: small distillation weight is the safer choice.')

## Cell 7 — Generation samples from each student

Same prompt, same seed, different α. Useful for the paper figure: shows whether
the BPC numbers translate to perceptible quality differences in the output.

In [ ]:
PROMPT = 'my lord,'
GEN_SEED = 42
for alpha in SWEEP_ALPHAS:
    tag = 'baseline (NLL-only)' if alpha == 0.0 else f'α_distill={alpha}'
    student = results[alpha]['student']
    out = wz.generate.generate(
        student, tok, prompt=PROMPT, max_new_tokens=100,
        block_size=BLOCK_SIZE, temperature=0.7, top_k=10,
        seed=GEN_SEED, device=device, use_hard=True,
    )
    print(f'\n===== {tag}  BPC {results[alpha]["bpc"]:.3f} =====')
    print(out)

## Cell 8 — Save sweep results

One consolidated artifact with everything needed for the paper figure.

In [ ]:
summary = {
    'teacher_bpc': teacher_bpc,
    'teacher_val': t_best['val'],
    'baseline_bpc': baseline_bpc,
    'sweep': {
        a: {'val': results[a]['val'], 'bpc': results[a]['bpc'],
            'history': results[a]['history']}
        for a in SWEEP_ALPHAS
    },
    'avg_cpt': avg_cpt,
    'config': {
        'student_d': S_D, 'student_L': S_LAYERS, 'vocab': VOCAB_SIZE,
        'block': BLOCK_SIZE, 'steps': S_STEPS, 'lr': LR, 'seed': SEED,
        'distill_temp': DIST_TEMP,
    },
}
out = RUN_DIR / 'nb18_sweep_summary.pt'
torch.save(summary, out)
print(f'saved → {out}  ({out.stat().st_size/1024:.1f} KB)')
print(f'\nDownload from /kaggle/working/runs/: nb18_sweep_summary.pt + student_alpha*.pt')

## Post-mortem

After running, paste back:
1. The 4 BPC values (baseline + 3 α)
2. The verdict line from Cell 6
3. The generation samples from Cell 7 — paper figure source

Based on the verdict, I'll:
- Update findings.md: either upgrade O1 → F17 (positive/negative/mixed) or retract O1
- Add the BPC-vs-α figure to the paper figures pile
- Drop the sweep table into the contributions list